# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"Published: {getattr(metadata, 'datePublished', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List out all record sets and their field ids for exploration
record_sets = getattr(metadata, 'recordSet', [])
if not isinstance(record_sets, list):
    record_sets = [record_sets]

print("Record sets and their fields (@id):\n")
record_set_ids = []
fields_by_record_set = {}

for record_set in record_sets:
    if hasattr(record_set, '@id'):
        rset_id = getattr(record_set, '@id')
    else:
        # If the dict style
        rset_id = record_set.get('@id', None)
    record_set_ids.append(rset_id)
    # Print RecordSet summary
    print(f"RecordSet: {rset_id}")
    # Now, list its fields
    fields = getattr(record_set, 'field', getattr(record_set, 'fields', []))
    if not fields:
        # Try as a dict
        fields = record_set.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    fields_ids = []
    for field in fields:
        fid = getattr(field, '@id', field.get('@id', None))
        fields_ids.append(fid)
        print(f"    Field: {fid}")
    fields_by_record_set[rset_id] = fields_ids
if not record_set_ids:
    print('No record sets found in metadata. It is possible record sets are defined implicitly or nested deeper in the schema.')


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If no record sets were found, try to infer available record sets from the mlcroissant API
# List of discovered record sets from the dataset
if record_set_ids:
    discovered_record_sets = record_set_ids
else:
    # Attempt to enumerate record sets based on the dataset instance
    discovered_record_sets = []
    try:
        # mlcroissant datasets usually have a .record_set_ids property for listing
        discovered_record_sets = list(dataset.record_set_ids)
    except Exception as e:
        print(f"Could not retrieve record set ids: {e}")

# If still not found, attempt to call records() and list all keys
if not discovered_record_sets:
    try:
        tmp = list(dataset.records())
        if tmp:
            discovered_record_sets = list(tmp[0].keys())
    except Exception as e:
        print(f"Could not enumerate keys from records(): {e}")

# For example, the FAIR2 Croissant schema usually contains one main data RecordSet. For this dataset, let's use the known RecordSet id directly as an illustrative example:
record_sets_to_load = []
# In real situations, the record set id may look like a URL or a string such as 'https://api.app.sen.science/frontiers/7154140/recordset-1'
# As the provided record sets list is empty, we can try with None, which loads the default one
record_sets_to_load = [None]
dataframes = {}

for rset in record_sets_to_load:
    print(f"\nLoading RecordSet: {rset if rset else 'default'}...")
    records = list(dataset.records(record_set=rset))
    print(f"Loaded {len(records)} records.")
    df = pd.DataFrame(records)
    dataframes[rset if rset else 'default'] = df

print("\nColumns in loaded DataFrame:")
print(dataframes['default'].columns.tolist())
dataframes['default'].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
df = dataframes['default']
print("Columns detected:", df.columns.tolist())
# Let's assume columns like 'coverage', 'peptide_count', or 'MW' are present (these are typical for proteomics tables)
# Replace with actual column names if needed
numeric_candidates = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
print(f"\nNumeric field candidates: {numeric_candidates}")
# For illustration, let's pick 'coverage' if it exists, otherwise the first numeric column
if 'coverage' in df.columns:
    numeric_field = 'coverage'
elif numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # Try to coerce any field named like 'Coverage' or 'MW'
    for attempt in ['Coverage', 'MW', 'molecular_weight', 'PeptideCount']:
        if attempt in df.columns:
            numeric_field = attempt
            break
    else:
        numeric_field = None

if numeric_field is None:
    print("No numeric field found for analysis. Please inspect the DataFrame.")
else:
    print(f"Selected numeric field for analysis: {numeric_field}")

    # Choose a threshold (e.g., mean or median to filter)
    threshold = df[numeric_field].median() if df[numeric_field].dtype in ['float64','int64'] else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, normalized_col]].head())

    # Group by a likely categorical field
    candidate_group_fields = [col for col in df.columns if ('description' in col.lower() or 'sample' in col.lower() or 'type' in col.lower())]
    group_field = candidate_group_fields[0] if candidate_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.